# BirdCLEF 2026: Dual-Pipeline Architecture and Ensemble Methodology

Kaggle银牌辅导2k：goodlucktrymore（wx） 
This notebook executes a complex ensembling strategy, combining a frame-wise Sound Event Detection (SED) model with a sequence-aware Prototypical State Space Model (ProtoSSMv5). The configuration dictionary `solutions` acts as the master controller, directing the instantiation, scaling, and blending of the models prior to generating the final submission.

In [1]:
solutions = {
 'type_add' :'single',
 'Models'   : [
  {'Model':'Model_7','subm':'submission.csv','weight':1, 'xSED':[],'LB':'0.948'}
 ]
}

In [2]:
_ensemble_models = [model['Model' ] for model in solutions['Models']]
_files_subm      = [model['subm'  ] for model in solutions['Models']]
_weights         = [model['weight'] for model in solutions['Models']]
_xsed            = [model['xSED'  ] for model in solutions['Models']]
_lbs             = [model['LB'    ] for model in solutions['Models']]

_single_solution = True if solutions['type_add']=='single' else False

## Pipeline 1: Distilled SED (Model_2)

Kaggle银牌辅导2k：goodlucktrymore（wx） 
This block defines our first core model, an efficient CNN-based SED architecture that leverages Teacher-Student distillation. Since Kaggle environments can struggle with heavy full-pipeline training, this is optimized for fast inference while maintaining high feature resolution.

**Architecture Highlights:**
* **Backbone:** `tf_efficientnet_b0.ns_jft_in1k` processes Mel Spectrograms (256 mels, n_fft=2048).
* **Generalized Mean (GeM) Frequency Pooling:** Replaces standard pooling over the frequency axis. A learnable parameter $p$ (initialized at 3.0) allows the network to dynamically sharpen focus on specific frequency bands characteristic of bird vocalizations.
* **Attention Bottleneck:** Features pass through a 512-dim dense layer followed by 1D Convolutions. This generates attention weights that aggregate frame-wise logits into robust clip-level predictions.
* **Perch Distillation Head:** A specialized branch uses Global Average Pooling (GAP) and a linear projection to map the EfficientNet features directly into the 1536-dimensional embedding space of Google's frozen Perch v2 model.

## Advanced Data Augmentation Strategy
Kaggle银牌辅导2k：goodlucktrymore（wx） 
To ensure the SED model generalizes well to unseen soundscapes and rare taxa, we implement a multi-tiered augmentation pipeline:

* **Signal-Level Noise:** Random gain jitter ($\pm$ 6.0 dB) and noise injection (SNR 10-30 dB).
* **SpecAugment:** Frequency and Time masking applied directly on the GPU.
* **Advanced MixUp (CutMix Hybrid):** We heavily utilize Beta-distribution blending. The pipeline performs **Focal-Focal MixUp** (blending two primary recordings) and **Focal-Soundscape MixUp** (injecting focal calls into labeled soundscape backgrounds). Rare species are dynamically upsampled so that every class has a guaranteed minimum of 20 samples per fold.

In [3]:
!python /kaggle/input/datasets/zihengedie/mycodeuse/my_code.py

ONNX Runtime available
2026-06-03 19:13:49.980503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780514030.233779      37 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780514030.299189      37 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780514030.869121      37 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780514030.869189      37 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780514030.869195      37 computation_placer.cc:177]